In [2]:
"""
PIPELINE STAGE: Carbon Impact Modeling, CEEI Computation & Feature Re-Engineering (Energy–Climate–Socioeconomic Analytics)

INPUT:
    processed_data/steps/11_Cap_Gen_Sun_Rain_Wind.xlsx

OUTPUT:
    processed_data/final/12_final.xlsx

LIBRARIES: pandas, os, calendar

1. OBJECTIVE:
    Compute advanced sustainability indicators from the integrated energy–climate–population dataset.
    This stage focuses on:
        - Source-based avoided CO2 emissions estimation
        - Carbon Eco-Efficiency Index (CEEI) calculation
        - Feature engineering and scientific variable compression
        - Final dataset structuring for ML and academic analysis

2. DATA INGESTION:
    A. Input Dataset:
       processed_data/steps/11_Cap_Gen_Sun_Rain_Wind.xlsx

       Contains:
           - Energy capacity (Biomass, Solar, Hydropower, Wind)
           - Electricity generation (same sources)
           - Population
           - Meteorological variables (Rainfall, Solar Radiation, Temperature, Wind Speed)
           - Spatial identifiers (Plate, Province)
           - Temporal identifiers (Year, Month)

3. CARBON EMISSIONS MODELING:
    A. Emission Factors:
       - Solar/Wind:
           EMISSION_FACTOR_SOLAR_WIND = 0.6242
       - Other renewables (Biomass, Hydropower):
           EMISSION_FACTOR_OTHER_REN = 0.5350

    B. Avoided Emissions Computation:
       For each energy source:

           A_source = Generation_source × Emission_Factor

       Outputs:
           - A_Biomass
           - A_Solar
           - A_Hydropower
           - A_Wind

    C. Total Avoided Emissions:

           T_Emission = Sum(all A_sources)

4. CARBON ECO-EFFICIENCY INDEX (CEEI):
    A. Temporal Energy Potential Modeling:
       Compute monthly theoretical generation capacity using:

           total_capacity × monthly_hours

    B. Monthly Hours Calculation:
       Using calendar month structure:

           month_hours = days_in_month × 24

    C. Actual Generation:

           total_generation = sum(all generation sources)

    D. CEEI Formula:

           CEEI = total_generation / theoretical_max_generation

    E. Constraints:
       - Avoid division by zero
       - Clip values to maximum 1.0

5. FEATURE ENGINEERING:
    A. Aggregated Variables:
       - total_capacity
       - total_generation

    B. Temporary Variables Removal:
       - month_hours
       - theoretical_max_gen

6. FEATURE SIMPLIFICATION (RENAME OPERATION):
    A. Capacity Features:
        capacity_Biomass      → C_Biomass
        capacity_Solar        → C_Solar
        capacity_Hydropower   → C_Hydropower
        capacity_Wind         → C_Wind

    B. Generation Features:
        generation_Biomass    → G_Biomass
        generation_Solar      → G_Solar
        generation_Hydropower → G_Hydropower
        generation_Wind       → G_Wind

    C. Emission Features:
        A_*                   → Avoided emissions per source
        T_Emission           → Total avoided emissions

7. DATA STRUCTURING:
    A. Final Column Order:

        Plate, Year, Month,
        Population,
        Climate Variables:
            Rainfall, Solar_Radiation, Temperature, Wind_Speed,
        Capacity Variables:
            C_Biomass, C_Solar, C_Hydropower, C_Wind,
        Generation Variables:
            G_Biomass, G_Solar, G_Hydropower, G_Wind,
        Emission Variables:
            A_Biomass, A_Solar, A_Hydropower, A_Wind,
            T_Emission,
        Sustainability Index:
            CEEI

8. OUTPUT GENERATION:
    A. Final Dataset:

           processed_data/final/12_final.xlsx

    B. Format:
       - Excel (.xlsx)
       - No index column

9. PROCESS LOGGING:
    A. Runtime Outputs:
       - Completion confirmation
       - Column structure validation
       - File path confirmation

10. EXPECTED OUTPUT DATASET:
    The final dataset enables:

        - Carbon impact assessment
        - Renewable energy efficiency modeling
        - Spatio-temporal sustainability analysis
        - Machine learning forecasting (energy + climate + emissions)
        - Policy-level decarbonization studies
"""

import pandas as pd
import os
import calendar

# =====================================================
# DOSYA YOLLARI
# =====================================================
base_dir = r"C:\Users\w11\dergi2"

# Dosya yolları oluşturuluyor
input_file = os.path.join(base_dir, "processed_data", "steps", "11_cap_gen_sun_rain_wind.xlsx")
output_file = os.path.join(base_dir, "processed_data", "final", "12_final.xlsx")

# Veriyi oku
df = pd.read_excel(input_file)

# =====================================================
# 1. ÖNLENEN EMİSYON HESAPLAMASI (Kaynak Bazlı)
# =====================================================
# Kaynak bazlı birleşik marj emisyon faktörleri (tCO2/MWh)
EMISSION_FACTOR_SOLAR_WIND = 0.6242
EMISSION_FACTOR_OTHER_REN = 0.5350 

print(">>> Kaynak bazlı önlenen CO2 Emisyonları hesaplanıyor...")

# Her kaynak için ayrı hesaplama
df['avoided_emissions_Biomass_tCO2'] = df['generation_Biomass'] * EMISSION_FACTOR_OTHER_REN
df['avoided_emissions_Solar_tCO2'] = df['generation_Solar'] * EMISSION_FACTOR_SOLAR_WIND
df['avoided_emissions_Hydropower_tCO2'] = df['generation_Hydropower'] * EMISSION_FACTOR_OTHER_REN
df['avoided_emissions_Wind_tCO2'] = df['generation_Wind'] * EMISSION_FACTOR_SOLAR_WIND

# Toplam önlenen emisyon hesabı
df['avoided_emissions_total_tCO2'] = (
    df['avoided_emissions_Biomass_tCO2'] + 
    df['avoided_emissions_Solar_tCO2'] + 
    df['avoided_emissions_Hydropower_tCO2'] + 
    df['avoided_emissions_Wind_tCO2']
)

# =====================================================
# 2. KARBON EKO-VERİMLİLİK İNDEKSİ (CEEI)
# =====================================================
def aydaki_saat_sayisini_bul(yil, ay):
    # İlgili ayın gün sayısını bulup 24 ile çarparak saat sayısını elde ederiz
    gun_sayisi = calendar.monthrange(int(yil), int(ay))[1]
    return gun_sayisi * 24

# Geçici değişkenler tanımlanıyor (DataFrame'e sütun olarak eklenmeden)
total_capacity = (df['capacity_Biomass'] + df['capacity_Solar'] + 
                  df['capacity_Hydropower'] + df['capacity_Wind'])

total_generation = (df['generation_Biomass'] + df['generation_Solar'] + 
                    df['generation_Hydropower'] + df['generation_Wind'])

# Ay bazlı saat hesabı ve teorik maksimum üretim
df['month_hours'] = df.apply(lambda row: aydaki_saat_sayisini_bul(row['Year'], row['Month']), axis=1)
df['theoretical_max_gen'] = total_capacity * df['month_hours']

# CEEI hesaplama (0'a bölme hatasını engellemek için maskeleme)
df['CEEI'] = 0.0
mask = df['theoretical_max_gen'] > 0

# total_generation bir değişken olduğu için loc içerisine yazmak yerine maskelendi
df.loc[mask, 'CEEI'] = total_generation[mask] / df.loc[mask, 'theoretical_max_gen']

# Sonucu maksimum 1.0 ile sınırlandır (Eğer üretim kapasiteden fazlaysa)
df['CEEI'] = df['CEEI'].clip(upper=1.0)

# Geçici hesaplama sütunlarını veri setinden kaldır
df = df.drop(columns=['month_hours', 'theoretical_max_gen'])

# =====================================================
# 3. SÜTUN İSİMLERİNİ KISALTMA 
# =====================================================
print(">>> Sütun isimleri makale formatına uygun olarak kısaltılıyor...")

rename_dict = {
    'capacity_Biomass': 'C_Biomass',
    'capacity_Solar': 'C_Solar',
    'capacity_Hydropower': 'C_Hydropower',
    'capacity_Wind': 'C_Wind',
    'generation_Biomass': 'G_Biomass',
    'generation_Solar': 'G_Solar',
    'generation_Hydropower': 'G_Hydropower',
    'generation_Wind': 'G_Wind',
    'avoided_emissions_Biomass_tCO2': 'A_Biomass',
    'avoided_emissions_Solar_tCO2': 'A_Solar',
    'avoided_emissions_Hydropower_tCO2': 'A_Hydropower',
    'avoided_emissions_Wind_tCO2': 'A_Wind',
    'avoided_emissions_total_tCO2': 'T_Emission'
}

df = df.rename(columns=rename_dict)

# =====================================================
# 4. SÜTUNLARI İSTENEN SIRAYA DİZME
# =====================================================
print(">>> Sütunlar belirlenen sıraya diziliyor...")

istenen_sira = [
    'Plate', 'Year', 'Month', 'Population', 'Rainfall', 'Solar_Radiation', 
    'Temperature', 'Wind_Speed', 'C_Biomass', 'C_Solar', 'C_Hydropower', 
    'C_Wind', 'G_Biomass', 'G_Solar', 'G_Hydropower', 'G_Wind', 
    'A_Biomass', 'A_Solar', 'A_Hydropower', 'A_Wind', 'T_Emission', 'CEEI'
]

# Veri setini sadece bu sıradaki sütunları içerecek şekilde güncelliyoruz
df = df[istenen_sira]

# =====================================================
# 5. KAYDET VE KONTROL ET
# =====================================================
df.to_excel(output_file, index=False)

print("\n>>> İŞLEM TAMAMLANDI!")
print(f"Dosya kaydedildi: {output_file}")
print("\nVeri setinizin yeni başlıkları ve sıralaması:")
print(df.columns.tolist())

>>> Kaynak bazlı önlenen CO2 Emisyonları hesaplanıyor...
>>> Sütun isimleri makale formatına uygun olarak kısaltılıyor...
>>> Sütunlar belirlenen sıraya diziliyor...

>>> İŞLEM TAMAMLANDI!
Dosya kaydedildi: C:\Users\w11\dergi2\processed_data\final\12_final.xlsx

Veri setinizin yeni başlıkları ve sıralaması:
['Plate', 'Year', 'Month', 'Population', 'Rainfall', 'Solar_Radiation', 'Temperature', 'Wind_Speed', 'C_Biomass', 'C_Solar', 'C_Hydropower', 'C_Wind', 'G_Biomass', 'G_Solar', 'G_Hydropower', 'G_Wind', 'A_Biomass', 'A_Solar', 'A_Hydropower', 'A_Wind', 'T_Emission', 'CEEI']
